# Divide-and-Conquer Design



```{contents}
:local:
:depth: 2
```


Divide-and-conquer algorithms solve a problem in three phases: divide the input into smaller parts, solve the smaller parts, and combine their results. Recursion is a natural implementation technique because each subproblem has the same shape as the original problem.


## The Pattern

```{index} The Pattern
```

A divide-and-conquer method usually follows this shape:

1. If the problem is small enough, solve it directly.
2. Divide the input into smaller pieces.
3. Recursively solve the pieces.
4. Combine the partial answers.

In [ ]:
using System;
using System.Linq;

foreach(int[] values in new[]{new[]{3,9,2,7,5},new[]{-8},Array.Empty<int>()})
{
 try{Console.WriteLine($"maximum={Max(values)}");}
 catch(ArgumentException){Console.WriteLine("empty maximum rejected");}
}
int Max(int[] values)
{
 ArgumentNullException.ThrowIfNull(values);
 if(values.Length==0)throw new ArgumentException("Nonempty input required.");
 return Solve(0,values.Length);
 int Solve(int start,int end)
 {
  if(end-start==1)return values[start];
  int middle=start+(end-start)/2;
  return Math.Max(Solve(start,middle),Solve(middle,end));
 }
}

The helper promise is the maximum of the nonempty half-open range [start,end). The wrapper rejects empty input before recursion. For length>1, both child ranges are nonempty, smaller, disjoint, and together cover the parent, including odd lengths. Inductively correct child maxima combine to the parent maximum. The size measure end−start decreases. T(n)=T(floor(n/2))+T(ceil(n/2))+Θ(1), hence Θ(n) work and Θ(log n) stack. A loop also takes Θ(n) work with less stack: splitting alone does not make maximum selection faster.

## Halving Strategies

```{index} Halving Strategies
```

Some algorithms solve only one half after dividing. Binary search is the classic example: compare with the middle value, then search the half where the target could still be.

In [ ]:
using System;
using System.Linq;

int[] sorted={2,4,6,8,10,12,14};
foreach(int target in new[]{2,10,14,5})Console.WriteLine($"target={target}, index={Search(sorted,target)}");
Console.WriteLine($"empty={Search(Array.Empty<int>(),5)}");
int Search(int[] values,int target)
{
 ArgumentNullException.ThrowIfNull(values);
 return Find(0,values.Length);
 int Find(int start,int end)
 {
  if(start==end)return -1;
  int middle=start+(end-start)/2;
  if(values[middle]==target)return middle;
  return target<values[middle]?Find(start,middle):Find(middle+1,end);
 }
}

Precondition: input is sorted in ascending order and is not modified during the search. The result is any matching index, or −1 when absent; duplicates do not promise a first match. The candidate interval is [start,end). Sortedness justifies discarding the impossible side. Excluding middle from the right interval prevents repeating a one-item range. Each surviving range is smaller; an empty range stops recursion. Only one child is explored, giving T(n)=T(floor(n/2))+Θ(1) as a size model, Θ(log n) worst-case time/stack, and Θ(1) best-case time. Checking sortedness on every call would add linear work.

## Combining Sorted Parts

```{index} Combining Sorted Parts
```

Merge sort depends on a combine step called **merge**. Given two sorted arrays, merge walks through both arrays and builds one sorted result.

In [ ]:
using System;
using System.Linq;

int[] left={1,4,4,9},right={2,4,10};
Console.WriteLine(string.Join(", ",StableSort.Merge(left,right,(a,b)=>a.CompareTo(b))));
Console.WriteLine($"empty merge length={StableSort.Merge(Array.Empty<int>(),Array.Empty<int>(),(a,b)=>a.CompareTo(b)).Length}");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


Merge requires two inputs sorted under the same comparison, and returns a fresh sorted array containing every input occurrence. At the main loop head, k=i+j and result[0,k) is the stable sorted merge of the consumed prefixes. The next smallest unconsumed item must be at one of the two fronts. Choosing the left front on equality preserves original order. Once one side is exhausted, copy the other remainder without further comparisons. For total N>0, merging takes Θ(N) time/output space and at most N−1 key comparisons; empty sides can need zero comparisons but still require copying.

## Merge Sort

```{index} Merge Sort
```

Merge sort divides the array into halves until each piece has zero or one item, then merges sorted pieces back together.

In [ ]:
using System;

int[] values = { 8, 3, 5, 1, 9, 2 };
int[] sorted = MergeSort(values);

Console.WriteLine(string.Join(", ", sorted));

int[] MergeSort(int[] values)
{
    if (values.Length <= 1)
    {
        return (int[])values.Clone();
    }

    int middle = values.Length / 2;
    int[] left = values[..middle];
    int[] right = values[middle..];

    return Merge(MergeSort(left), MergeSort(right));
}

int[] Merge(int[] left, int[] right)
{
    int[] result = new int[left.Length + right.Length];
    int i = 0;
    int j = 0;
    int k = 0;

    while (i < left.Length && j < right.Length)
    {
        result[k++] = left[i] <= right[j] ? left[i++] : right[j++];
    }

    while (i < left.Length)
    {
        result[k++] = left[i++];
    }

    while (j < right.Length)
    {
        result[k++] = right[j++];
    }

    return result;
}


The copy-returning contract leaves the input unchanged and returns a fresh array even for zero or one item. Slices allocate new arrays, making this version easy to read but allocation-heavy. T(n)=T(floor(n/2))+T(ceil(n/2))+Θ(n), so time is Θ(n log n) for n≥2. Stack depth is Θ(log n). The next section replaces slices with index ranges and a reusable buffer; it also verifies stability and preservation. Comparisons are only one part of sorting cost: copying still happens when a side is exhausted.

## Design Checklist

```{index} Design Checklist
```

When designing a divide-and-conquer algorithm, ask:

1. What is the smallest case I can solve directly?
2. How do I divide the input?
3. Do I solve one subproblem or multiple subproblems?
4. How do I combine the answers?
5. What recurrence describes the cost?

## Exercise

```{index} Exercise
```

Merge two sorted invoice batches, including duplicates and an empty side. Verify the exact expected sequence, not just increasing order.

In [ ]:
using System;
using System.Linq;

int[] left={1,3,3},right={2,3};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
int[] left={1,3,3},right={2,3};
int[] result=StableSort.Merge(left,right,(a,b)=>a.CompareTo(b));
if(!result.SequenceEqual(new[]{1,2,3,3,3}))throw new Exception("Merge mismatch.");
Console.WriteLine(string.Join(", ",result));
Console.WriteLine(string.Join(", ",StableSort.Merge(left,Array.Empty<int>(),(a,b)=>a.CompareTo(b))));
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


1, 2, 3, 3, 3
1, 3, 3


```{rubric} Footnotes
```
[^1]: Divide-and-conquer is not automatically faster than iteration. It pays off when the split and combine structure reduces work or clarifies a naturally recursive problem.
[^2]: The range convention `[start, end)` includes `start` and excludes `end`. It is common in C# because the length of the range is `end - start`.
